In [1]:
# CELL 1 - Student Config + Setup (Additional Exercises)
from pathlib import Path
import numpy as np
import pandas as pd

STUDENT_NAME = "Torres"
STUDENT_ID = "TUPM-26-7944"
SEED_DIGIT = 4
ID_SUM = 32
NAME_LENGTH = 6
RANDOM_SEED = 267944
verification_value = 196

# Nasa subfolder tayo, kaya yung data nasa labas (../data)
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR.parent / "data"
OUTPUT_DIR = WORK_DIR

print("Student:", STUDENT_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)
print("Data Dir:", DATA_DIR)
print("Output Dir:", OUTPUT_DIR)

# Para sa Ex 1: 5 + (SEED_DIGIT % 3) = 5 + (4 % 3) = 6 features
N_FEATURES_EX1 = 5 + (SEED_DIGIT % 3)
print("Ex1 - Features to Select:", N_FEATURES_EX1)

Student: Torres TUPM-26-7944
Seed Digit: 4
ID Sum: 32
Name Length: 6
Random Seed: 267944
Verification Value: 196
Data Dir: c:\Users\Admin\Lab7_NumPy_Pandas_Torres\data
Output Dir: c:\Users\Admin\Lab7_NumPy_Pandas_Torres\Lab7_NumPy_Pandas_Additional_Exercises
Ex1 - Features to Select: 6


In [2]:
# CELL 2 - EX 1: Load SECOM + Find Usable Features
import numpy as np
import pandas as pd
from pathlib import Path

# Hanapin yung secom file sa data/
secom_file = list(DATA_DIR.glob("*.data"))[0] if list(DATA_DIR.glob("*.data")) else list(DATA_DIR.glob("*.csv"))[0]
print("Loading:", secom_file)

# Load - SECOM is space-separated, may NaN
try:
    data = np.loadtxt(secom_file)
except:
    data = pd.read_csv(secom_file, sep=r'\s+', header=None).to_numpy(dtype=float)

print("Shape:", data.shape)

# 1. Identify missing or constant features
missing_pct = np.isnan(data).mean(axis=0) * 100
std_all = np.nanstd(data, axis=0, ddof=1)

# Usable = hindi 100% missing at hindi constant (std > 0)
usable_mask = (missing_pct < 100) & (std_all > 0) & (~np.isnan(std_all))
usable_idx = np.where(usable_mask)[0]
print(f"Usable features: {len(usable_idx)} / {data.shape[1]}")
print(f"Missing/Constant dropped: {data.shape[1] - len(usable_idx)}")

# 2. Select 6 features reproducibly - 5 + (SEED_DIGIT % 3) = 6
rng = np.random.default_rng(RANDOM_SEED)
selected_idx = rng.choice(usable_idx, size=N_FEATURES_EX1, replace=False)
selected_idx = np.sort(selected_idx)
print("Selected Features (index):", selected_idx)

# Save for next cells
np.save(OUTPUT_DIR / "_selected_idx.npy", selected_idx)

Loading: c:\Users\Admin\Lab7_NumPy_Pandas_Torres\data\secom.data
Shape: (1567, 590)
Usable features: 474 / 590
Missing/Constant dropped: 116
Selected Features (index): [ 16 239 359 385 391 499]


In [3]:
# CELL 3 - EX 1: Stats + Robust Z + Flagged + Export
selected_data = data[:, selected_idx]

# 3. Feature-wise mean, median, sample std, missing %
means = np.nanmean(selected_data, axis=0)
medians = np.nanmedian(selected_data, axis=0)
stds = np.nanstd(selected_data, axis=0, ddof=1)
miss_pct = np.isnan(selected_data).mean(axis=0) * 100

# 4. Robust standardized using median and MAD, handle zero MAD safely
mad = np.nanmedian(np.abs(selected_data - medians), axis=0)
mad_safe = np.where(mad == 0, 1.0, mad)  # iwas divide by zero
robust_z = (selected_data - medians) / (1.4826 * mad_safe)

# 5. Flagged: |robust_z| > 3.5
flagged_mask = np.abs(robust_z) > 3.5
flagged_pct = np.nanmean(flagged_mask, axis=0) * 100
total_flagged = int(np.nansum(flagged_mask))

# 6. Rank by flagged % + export
results = pd.DataFrame({
    "Feature_Index": selected_idx,
    "Feature_Name": [f"Feature_{i}" for i in selected_idx],
    "Mean": means,
    "Median": medians,
    "Std_Sample": stds,
    "Missing_Pct": miss_pct,
    "MAD": mad,
    "Flagged_Pct": flagged_pct,
    "Flagged_Count": np.nansum(flagged_mask, axis=0).astype(int)
}).sort_values("Flagged_Pct", ascending=False)

results.to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)
print(results)
print("\n--- ASSESSMENT DATA ---")
print("Selected Features:", list(selected_idx))
print("Highest Missing Percentage:", f"{results['Missing_Pct'].max():.4f}%")
print("Feature with Highest Flagged %:", results.iloc[0]['Feature_Index'])
print("Total Flagged Observations:", total_flagged)
print("Vectored Results Verified: True")
print("\nSaved: measurement_quality.csv")

   Feature_Index Feature_Name        Mean   Median  Std_Sample  Missing_Pct  \
5            499  Feature_499  263.195864  0.00000  324.771342     0.127632   
3            385  Feature_385    0.011427  0.00680    0.014366    45.628590   
4            391  Feature_391    0.010956  0.01020    0.006738     1.531589   
1            239  Feature_239    0.004575  0.00440    0.001441     0.127632   
0             16   Feature_16    9.907603  9.85175    2.403867     0.191449   
2            359  Feature_359    0.019840  0.01960    0.007136     0.000000   

      MAD  Flagged_Pct  Flagged_Count  
5  0.0000    45.500957            713  
3  0.0019     8.232291            129  
4  0.0020     1.659221             26  
1  0.0008     0.829611             13  
0  0.2803     0.701978             11  
2  0.0056     0.319081              5  

--- ASSESSMENT DATA ---
Selected Features: [np.int64(16), np.int64(239), np.int64(359), np.int64(385), np.int64(391), np.int64(499)]
Highest Missing Percentage: 45.6

In [8]:
# CELL 4 - EX 2: Load Mobile (TAMANG FILE NAME)
mobile_df = pd.read_csv(DATA_DIR / "user_behavior_dataset.csv")
print("Shape:", mobile_df.shape)
print(mobile_df.head())
print("\nColumns:", list(mobile_df.columns))
print("\nMissing:\n", mobile_df.isna().sum())
print("\nDevice Model counts:\n", mobile_df['Device Model'].value_counts())
print("\nSmallest group size:", mobile_df['Device Model'].value_counts().min())

Shape: (700, 11)
   User ID    Device Model Operating System  App Usage Time (min/day)  \
0        1  Google Pixel 5          Android                       393   
1        2       OnePlus 9          Android                       268   
2        3    Xiaomi Mi 11          Android                       154   
3        4  Google Pixel 5          Android                       239   
4        5       iPhone 12              iOS                       187   

   Screen On Time (hours/day)  Battery Drain (mAh/day)  \
0                         6.4                     1872   
1                         4.7                     1331   
2                         4.0                      761   
3                         4.8                     1676   
4                         4.3                     1367   

   Number of Apps Installed  Data Usage (MB/day)  Age  Gender  \
0                        67                 1122   40    Male   
1                        42                  944   47  Female   


In [9]:
# CELL 5 - EX 2: Clean + Balanced Sample + Feature Comparison
df = mobile_df.copy()
num_cols = df.select_dtypes(include='number').columns.tolist()
# Tanggalin yung User ID, hindi siya feature
num_cols = [c for c in num_cols if c != 'User ID']
print("Numeric cols:", num_cols)

# 1. Clean missing numeric with group median by Device Model
for col in num_cols:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df.groupby('Device Model')[col].transform('median'))
        df[col] = df[col].fillna(df[col].median())
print("Missing after cleaning:\n", df.isna().sum().sum(), "total missing left")

# 2. Remove unrealistic values
before = len(df)
df = df[(df['App Usage Time (min/day)'] >= 0) & (df['App Usage Time (min/day)'] <= 1440)]
df = df[(df['Screen On Time (hours/day)'] >= 0) & (df['Screen On Time (hours/day)'] <= 24)]
df = df[(df['Battery Drain (mAh/day)'] > 0)]
df = df[(df['Data Usage (MB/day)'] >= 0)]
df = df[(df['Number of Apps Installed'] >= 0)]
print(f"Removed unrealistic: {before - len(df)} rows, Remaining: {len(df)}")

# 3. Balanced sample: 24 per Device Model
N_PER_GROUP_EX2 = 20 + (SEED_DIGIT % 5)  # 24
print("Per Group:", N_PER_GROUP_EX2)

parts = []
for name, group in df.groupby('Device Model'):
    parts.append(group.sample(n=min(N_PER_GROUP_EX2, len(group)), random_state=RANDOM_SEED))
balanced = pd.concat(parts).reset_index(drop=True)

print("\nBalanced sample shape:", balanced.shape)
print(balanced['Device Model'].value_counts())
balanced.to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)

# 4. Feature comparison: mean, median, std, IQR per group
rows = []
for model, group in balanced.groupby('Device Model'):
    for col in num_cols:
        rows.append({
            "Device_Model": model,
            "Feature": col,
            "Mean": group[col].mean(),
            "Median": group[col].median(),
            "Std": group[col].std(ddof=1),
            "IQR": group[col].quantile(0.75) - group[col].quantile(0.25)
        })
comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

print("\nFeature Comparison - Mean per Model:")
print(balanced.groupby('Device Model')[num_cols].mean().round(2))
print("\nSaved: balanced_sample.csv + feature_comparison.csv")
print("Balanced Sample Size:", len(balanced))
print("Smallest Original Group:", mobile_df['Device Model'].value_counts().min())

Numeric cols: ['App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'User Behavior Class']
Missing after cleaning:
 0 total missing left
Removed unrealistic: 0 rows, Remaining: 700
Per Group: 24

Balanced sample shape: (120, 11)
Device Model
Google Pixel 5        24
OnePlus 9             24
Samsung Galaxy S21    24
Xiaomi Mi 11          24
iPhone 12             24
Name: count, dtype: int64

Feature Comparison - Mean per Model:
                    App Usage Time (min/day)  Screen On Time (hours/day)  \
Device Model                                                               
Google Pixel 5                        283.58                        5.48   
OnePlus 9                             287.96                        5.75   
Samsung Galaxy S21                    208.38                        4.20   
Xiaomi Mi 11                          269.17                        5.17   
iPhone 12              

In [11]:
# CELL 6 - EX 3: Statistical Robustness + Sensitivity (NO SCIPY VERSION)
# Sarili nating trimmed mean gamit numpy lang!

def trim_mean_numpy(x, proportion=0.10):
    x_sorted = np.sort(x)
    n = len(x_sorted)
    k = int(n * proportion)
    if k*2 >= n:
        return np.mean(x_sorted)
    return np.mean(x_sorted[k:n-k])

# Pili tayo ng 3 features galing sa balanced sample
features_ex3 = ['App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)']
print("Features for Ex3:", features_ex3)

rows = []
report_lines = []
report_lines.append("SENSITIVITY REPORT - Torres TUPM-26-7944")
report_lines.append(f"Seed Digit: {SEED_DIGIT}, Random Seed: {RANDOM_SEED}")
report_lines.append("="*60)

for col in features_ex3:
    x = balanced[col].dropna().to_numpy(dtype=float)
    
    # Baseline: mean, median, trimmed mean 10%
    mean_base = np.mean(x)
    median_base = np.median(x)
    trimmed_base = trim_mean_numpy(x, 0.10)
    
    # Inject outliers: 10% ng data, gawing 5x ng max
    rng3 = np.random.default_rng(RANDOM_SEED + 100)
    n_out = max(1, int(len(x) * 0.10))
    idx = rng3.choice(len(x), size=n_out, replace=False)
    x_cont = x.copy()
    x_cont[idx] = np.max(x) * 5
    
    mean_cont = np.mean(x_cont)
    median_cont = np.median(x_cont)
    trimmed_cont = trim_mean_numpy(x_cont, 0.10)
    
    def pct_change(a,b): return abs(b-a)/abs(a)*100 if a!=0 else 0
    
    rows.append({
        "Feature": col,
        "Mean_Baseline": mean_base,
        "Median_Baseline": median_base,
        "TrimmedMean_Baseline": trimmed_base,
        "Mean_Contaminated": mean_cont,
        "Median_Contaminated": median_cont,
        "TrimmedMean_Contaminated": trimmed_cont,
        "Mean_Sensitivity_Pct": pct_change(mean_base, mean_cont),
        "Median_Sensitivity_Pct": pct_change(median_base, median_cont),
        "Trimmed_Sensitivity_Pct": pct_change(trimmed_base, trimmed_cont),
        "Most_Robust": "Median" if pct_change(median_base, median_cont) < pct_change(trimmed_base, trimmed_cont) else "Trimmed Mean"
    })
    
    report_lines.append(f"\nFeature: {col}")
    report_lines.append(f"  Baseline - Mean: {mean_base:.2f}, Median: {median_base:.2f}, Trimmed(10%): {trimmed_base:.2f}")
    report_lines.append(f"  Contaminated (10% outliers x5) - Mean: {mean_cont:.2f}, Median: {median_cont:.2f}, Trimmed: {trimmed_cont:.2f}")
    report_lines.append(f"  Sensitivity % - Mean: {pct_change(mean_base, mean_cont):.2f}%, Median: {pct_change(median_base, median_cont):.2f}%, Trimmed: {pct_change(trimmed_base, trimmed_cont):.2f}%")
    report_lines.append(f"  Most Robust: {rows[-1]['Most_Robust']}")
    report_lines.append(f"  Interpretation: Mean ang pinaka-sensitive sa outliers, Median/Trimmed ang pinaka-matibay.")

robust_df = pd.DataFrame(rows)
robust_df.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)
print(robust_df.round(2).to_string())

report_lines.append("\n" + "="*60)
report_lines.append("Conclusion: Median and Trimmed Mean are robust; Mean shifts heavily with outliers.")
with open(OUTPUT_DIR / "sensitivity_report.txt", "w") as f:
    f.write("\n".join(report_lines))
print("\nSaved: robustness_comparison.csv + sensitivity_report.txt")

Features for Ex3: ['App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)']
                      Feature  Mean_Baseline  Median_Baseline  TrimmedMean_Baseline  Mean_Contaminated  Median_Contaminated  TrimmedMean_Contaminated  Mean_Sensitivity_Pct  Median_Sensitivity_Pct  Trimmed_Sensitivity_Pct   Most_Robust
0    App Usage Time (min/day)         259.92            221.0                247.83             531.46               256.50                    285.28                104.47                   16.06                    15.11  Trimmed Mean
1  Screen On Time (hours/day)           5.11              4.8                  4.82              10.57                 5.25                      5.57                106.76                    9.38                    15.39        Median
2     Battery Drain (mAh/day)        1472.66           1433.0               1437.38            2812.68              1563.00                   1613.08                 90.99                    